# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR^2 dataset using the `mlcroissant` library, following the FAIR data principles in practice.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Accessing the name and description
print("Dataset loaded.")
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their field and column `@id`s. All entities in the Croissant schema are referenced by their `@id` field.

First, discover the record sets in the metadata.

In [ ]:
# List all record sets with their IDs and associated fields/columns
record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for idx, rset in enumerate(record_sets):
    print(f"Record Set {idx + 1}:")
    print(f"  @id: {rset.id}")
    print(f"  Name: {getattr(rset, 'name', '(no name)')}")
    print(f"  Description: {getattr(rset, 'description', '(no description)')}")
    # Fields
    try:
        fields = list(getattr(rset, 'fields', []))
        print(f"  Fields (by @id):")
        for field in fields:
            print(f"    - {getattr(field, 'id', field)} (name: {getattr(field, 'name', '')})")
    except Exception:
        print("    No fields found.")
    # Columns
    try:
        columns = list(getattr(rset, 'columns', []))
        print(f"  Columns (by @id):")
        for col in columns:
            print(f"    - {getattr(col, 'id', col)} (name: {getattr(col, 'name', '')})")
    except Exception:
        print("    No columns found.")
    print()

Below, we print the first 2 records from the main record set.

> Note: Update `main_record_set_id` to your dataset's main record set `@id` based on the overview above.

In [ ]:
# Choose the MAIN record set (edit this @id if necessary based on the above output)
main_record_set_id = None
if len(record_sets) > 0:
    main_record_set_id = record_sets[0].id
else:
    raise ValueError("No record sets found in dataset metadata.")

# Show first records in the chosen record set
for idx, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    print(rec)
    if idx >= 1:
        break

## 3. Data Extraction
Load data from record sets into Pandas DataFrames for analysis. Refer to record set and field/column `@id`s from the overview.

The below example loads all record sets, but you can filter to those of interest.

In [ ]:
# Extract all data from each record set
dataframes = {}
record_set_ids = [rset.id for rset in record_sets]

for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Check columns for the main record set
print(f"Main record set columns (@id): {dataframes[main_record_set_id].columns.tolist()}")
# Show a preview
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's explore numeric fields, filter records based on a threshold, normalize values, and optionally group by another field.

Select a numeric field (`@id`) and a group field (`@id`) from the column overview above.

*__You may need to change the default values below to match your dataset columns!__*

In [ ]:
# Set which numeric field and grouping field to analyze (by column name/@id)
# Please update these values to match your dataset columns/fields
# Example: numeric_field = 'Abundance', group_field = 'Sample_Name'

numeric_field = None
group_field = None
df = dataframes[main_record_set_id]

# Try to auto-detect numeric fields
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
if len(numeric_candidates) > 0:
    numeric_field = numeric_candidates[0]

# Try to pick a non-numeric field for grouping
group_candidates = [col for col in df.columns if col not in numeric_candidates]
if len(group_candidates) > 0:
    group_field = group_candidates[0]

print(f"Using numeric_field: {numeric_field}")
print(f"Using group_field: {group_field}")

if numeric_field is None:
    raise ValueError('No numeric field detected for analysis. Please set numeric_field manually.')

# Filtering
threshold = df[numeric_field].quantile(0.9)  # Take top 10% as example threshold
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
print(filtered_df.head())

# Normalizing
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping
if group_field is not None and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.sort_values(numeric_field, ascending=False).head())
else:
    print('No group field to aggregate by.')

## 5. Visualization
Visualize data distributions or relationships between selected fields. Below, we show a histogram and scatter plot for the main numeric fields.

> Edit the column names as needed for your profile fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=50, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

if numeric_field is not None and group_field is not None and group_field in df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field} (filtered)")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We loaded, explored, filtered, and visualized one or more record sets from the FAIR^2 dataset using `mlcroissant`.
- The record set(s) and fields/columns are identified by their Croissant schema `@id`s for reproducible workflows.
- Further downstream analysis, such as machine learning or statistical hypothesis testing, can now be performed on the processed DataFrames.

\n
Happy analyzing!
